# 🎵 Spotify 2023 — ETL & Análise Exploratória de Dados

![Python](https://img.shields.io/badge/Python-3.10+-3776AB?style=flat&logo=python&logoColor=white)
![Pandas](https://img.shields.io/badge/Pandas-2.0-150458?style=flat&logo=pandas&logoColor=white)
![Plotly](https://img.shields.io/badge/Plotly-5.18-3F4F75?style=flat&logo=plotly&logoColor=white)
![Status](https://img.shields.io/badge/Status-Completo-1DB954?style=flat)

> Desenvolvido por **[Sander Augusto Garcia](https://github.com/SanderAugustoGarcia)**  
> 🚀 **[Dashboard interactivo](https://edaspotify.streamlit.app)** — Streamlit

---

## Contexto

Análise exploratória das **952 músicas mais ouvidas no Spotify em 2023**, combinando atributos musicais técnicos (BPM, danceability, energy…) com métricas reais de negócio em 4 plataformas: Spotify, Apple Music, Deezer e Shazam.

## Estrutura

| Secção | Conteúdo |
|--------|----------|
| **1. ETL** | Carregamento, diagnóstico de qualidade, limpeza e feature engineering |
| **2. EDA — Q1** | Quão desigual é a distribuição de streams? |
| **3. EDA — Q2** | Existe sazonalidade nos lançamentos musicais? |
| **4. EDA — Q3** | Músicas antigas ainda dominam os charts de 2023? |
| **5. EDA — Q4** | Colaborações geram mais streams do que solos? |
| **6. Sumário** | Insights e próximos passos |


## 0. Imports e configuração

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings("ignore")

# ── Paleta Spotify ──────────────────────────────────────────────────────────
GREEN  = "#1DB954"
CORAL  = "#E8563A"
AMBER  = "#F59B23"
PURPLE = "#9B72CF"
BLUE   = "#4FC3F7"
GRAY   = "#B3B3B3"
BG     = "#121212"
BG2    = "#1E1E1E"

ERA_COLORS = {
    "Pré-2000":  "#B3B3B3",
    "2000s":     "#9B72CF",
    "2010s":     "#4FC3F7",
    "2020\u20132021": "#1DB954",
    "2022\u20132023": "#E8563A",
}

# ── Helper: aplica tema Spotify sem depender de templates ───────────────────
def spotify_layout(fig, title="", height=420, showlegend=False,
                   xaxis_title="", yaxis_title=""):
    fig.update_layout(
        title=dict(text=title, font=dict(color="white", size=15, family="Arial"), x=0.01),
        paper_bgcolor=BG,
        plot_bgcolor=BG2,
        font=dict(family="Arial", color=GRAY, size=12),
        xaxis=dict(gridcolor="#282828", linecolor="#282828",
                   tickcolor="#535353", zerolinecolor="#282828",
                   title_text=xaxis_title),
        yaxis=dict(gridcolor="#282828", linecolor="#282828",
                   tickcolor="#535353", zerolinecolor="#282828",
                   title_text=yaxis_title),
        legend=dict(bgcolor=BG2, bordercolor="#282828", borderwidth=1,
                    font=dict(color=GRAY)),
        margin=dict(l=50, r=30, t=55, b=50),
        hoverlabel=dict(bgcolor="#282828", bordercolor="#535353",
                        font=dict(color="white", size=12)),
        colorway=[GREEN, CORAL, AMBER, PURPLE, BLUE, GRAY],
        height=height,
        showlegend=showlegend,
    )
    return fig

print("✅ Imports e paleta Spotify configurados")

---
## 1. ETL — Extracção, Transformação e Carregamento

### 1.1 Carregar dados

In [ ]:
df_raw = pd.read_csv(
    "spotify-2023.csv",
    encoding="utf-8",
    encoding_errors="replace",
)
print(f"Dimensões: {df_raw.shape[0]} linhas × {df_raw.shape[1]} colunas")
df_raw.head(3)

### 1.2 Diagnóstico de qualidade

In [ ]:
missing   = df_raw.isnull().sum()
empty_str = (df_raw == "").sum()
issues    = (missing + empty_str).rename("total")

print("Colunas com problemas de qualidade:\n")
display(
    issues[issues > 0]
    .reset_index()
    .rename(columns={"index": "coluna"})
    .assign(nulos=missing[issues > 0].values,
            vazios=empty_str[issues > 0].values)
)

**Issues identificados e decisões de tratamento:**

| Coluna | Problema | Decisão |
|--------|----------|---------|
| `streams` | 1 linha corrompida — dados de BPM/key entraram na coluna errada | Remover a linha |
| `key` | 95 valores em branco | Substituir por `'Unknown'` |
| `in_shazam_charts` | 50 valores nulos | Substituir por `0` (não entrou nos charts) |

### 1.3 Limpeza

In [ ]:
bad_mask = pd.to_numeric(df_raw["streams"], errors="coerce").isna()
print(f"Linha removida: '{df_raw.loc[bad_mask, 'track_name'].values[0]}'")

df = df_raw[~bad_mask].copy()
df["streams"] = pd.to_numeric(df["streams"])

num_cols = [
    "artist_count", "released_year", "released_month", "released_day",
    "in_spotify_playlists", "in_spotify_charts",
    "in_apple_playlists", "in_apple_charts",
    "in_deezer_playlists", "in_deezer_charts", "in_shazam_charts",
    "bpm", "danceability_%", "valence_%", "energy_%",
    "acousticness_%", "instrumentalness_%", "liveness_%", "speechiness_%",
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["key"] = df["key"].replace("", np.nan).fillna("Unknown")
df["in_shazam_charts"] = df["in_shazam_charts"].fillna(0)

print(f"\n✅ Dataset limpo: {df.shape[0]} linhas × {df.shape[1]} colunas")

### 1.4 Feature engineering

In [ ]:
df["streams_M"] = df["streams"] / 1e6

df["era"] = pd.cut(
    df["released_year"],
    bins=[0, 1999, 2009, 2019, 2021, 2023],
    labels=["Pré-2000", "2000s", "2010s", "2020\u20132021", "2022\u20132023"],
).astype(str)

df["collab"] = df["artist_count"].apply(lambda x: "Colaboração" if x > 1 else "Solo")

print("Features criadas: streams_M, era, collab")
df[["track_name", "artist(s)_name", "released_year", "era", "collab", "streams_M"]].head(5)

### 1.5 Estatísticas descritivas

In [ ]:
desc = df["streams"].describe()
print("Streams — Estatísticas descritivas\n")
print(f"  Contagem : {int(desc['count']):,} músicas")
print(f"  Média    : {desc['mean']/1e6:>8.1f} M")
print(f"  Mediana  : {desc['50%']/1e6:>8.1f} M")
print(f"  Mínimo   : {desc['min']/1e6:>8.3f} M")
print(f"  Máximo   : {desc['max']/1e6:>8.1f} M")
print(f"  Std      : {desc['std']/1e6:>8.1f} M")
print()
print(f"\u26a0\ufe0f  Média ({desc['mean']/1e6:.0f}M) >> Mediana ({desc['50%']/1e6:.0f}M)")
print("   \u2192 distribuição fortemente assimétrica à direita")

---
## 2. Q1 — Quão desigual é a distribuição de streams?

**Hipótese:** O mercado musical segue uma lei de potência — poucos artistas concentram a maioria dos streams.  
**Métricas:** histograma, distribuição log₁₀, Curva de Lorenz, Coeficiente de Gini.

In [ ]:
s_sorted    = np.sort(df["streams"].values)
cum_streams = np.cumsum(s_sorted) / s_sorted.sum()
cum_pop     = np.arange(1, len(s_sorted) + 1) / len(s_sorted)
gini        = 1 - 2 * np.trapezoid(cum_streams, cum_pop)
idx_10      = int(0.9 * len(s_sorted))
share_top10 = (1 - cum_streams[idx_10]) * 100

print(f"Gini        = {gini:.3f}")
print(f"Top 10%     = {share_top10:.1f}% dos streams")
print(f"Média       = {df['streams_M'].mean():.0f}M")
print(f"Mediana     = {df['streams_M'].median():.0f}M")

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Distribuição em log₁₀", "Curva de Lorenz"],
)

# Panel 1 — Histograma log10
log_s = np.log10(df["streams"].clip(lower=1))
fig.add_trace(
    go.Histogram(x=log_s, nbinsx=40, marker_color=GREEN, opacity=0.85,
                 hovertemplate="log₁₀: %{x:.2f}<br>Músicas: %{y}<extra></extra>",
                 name="Distribuição"),
    row=1, col=1
)
fig.add_vline(x=float(np.log10(df["streams"].median())),
              line_dash="dash", line_color=CORAL, line_width=2,
              annotation_text=f"Mediana {df['streams_M'].median():.0f}M",
              annotation_font_color=CORAL, row=1, col=1)
fig.add_vline(x=float(np.log10(df["streams"].mean())),
              line_dash="dot", line_color=AMBER, line_width=2,
              annotation_text=f"Média {df['streams_M'].mean():.0f}M",
              annotation_font_color=AMBER, annotation_position="top left", row=1, col=1)

# Panel 2 — Lorenz
fig.add_trace(
    go.Scatter(x=cum_pop*100, y=cum_streams*100,
               mode="lines", line=dict(color=GREEN, width=2.5),
               fill="tonexty", fillcolor="rgba(29,185,84,0.1)",
               name="Curva de Lorenz",
               hovertemplate="Top %{x:.1f}% músicas = %{y:.1f}% streams<extra></extra>"),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=[0,100], y=[0,100], mode="lines",
               line=dict(color="#535353", width=1.2, dash="dash"),
               name="Igualdade perfeita"),
    row=1, col=2
)
fig.add_annotation(
    x=72, y=28, xref="x2", yref="y2",
    text=f"<b>Top 10% = {share_top10:.0f}% streams</b><br>Gini = {gini:.2f}",
    showarrow=True, arrowhead=2, ax=0, ay=-45,
    bgcolor=BG2, bordercolor=GREEN, borderwidth=1,
    font=dict(color=GREEN, size=12),
)

spotify_layout(fig, title="Q1 — Distribuição de streams", height=420, showlegend=True)
fig.update_xaxes(title_text="log₁₀(Streams)", row=1, col=1)
fig.update_yaxes(title_text="Nº de músicas", row=1, col=1)
fig.update_xaxes(title_text="% músicas (acumulado)", row=1, col=2)
fig.update_yaxes(title_text="% streams (acumulado)", row=1, col=2)
fig.show()

### 💡 Insights — Q1

| Métrica | Valor |
|---------|-------|
| Coeficiente de Gini | **0.53** |
| Top 10% das músicas | **37%** de todos os streams |
| Média | **514M** streams |
| Mediana | **291M** streams |

- A distribuição segue uma **lei de potência** — média quase o dobro da mediana, sinal de cauda longa extrema
- O **Gini de 0.53** é superior ao da desigualdade de rendimento em Portugal (~0.33)
- A transformação **log₁₀** aproxima os dados de uma normal — relevante para modelação futura
- **Implicação:** o algoritmo de playlists editoriais é o principal driver de sucesso

---
## 3. Q2 — Existe sazonalidade nos lançamentos musicais?

**Hipótese:** A indústria concentra lançamentos no início do ano e antes do Verão.  
**Nota:** Análise focada em 2022–2023 (anos com dados suficientemente densos).

In [ ]:
month_names = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]

df_recent = df[df["released_year"] >= 2022].copy()
m_count   = df_recent.groupby("released_month").size().reindex(range(1,13), fill_value=0)
m_streams = df_recent.groupby("released_month")["streams_M"].median().reindex(range(1,13), fill_value=0)

print(f"Mês com mais lançamentos  : {month_names[m_count.idxmax()-1]} ({m_count.max()} músicas)")
print(f"Mês com menos lançamentos : {month_names[m_count.idxmin()-1]} ({m_count.min()} músicas)")
print(f"Mês com mais streams (med): {month_names[m_streams.idxmax()-1]} ({m_streams.max():.0f}M)")

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Volume de lançamentos por mês", "Streams mediana por mês"],
)

colors_vol = [CORAL if v == m_count.max() else GREEN for v in m_count.values]
fig.add_trace(
    go.Bar(x=month_names, y=m_count.values, marker_color=colors_vol, opacity=0.9,
           text=m_count.values, textposition="outside",
           textfont=dict(color="white", size=11),
           hovertemplate="%{x}: <b>%{y} músicas</b><extra></extra>"),
    row=1, col=1
)
fig.add_hline(y=m_count.mean(), line_dash="dash", line_color="#535353",
              annotation_text=f"Média {m_count.mean():.0f}", row=1, col=1)

colors_str = [AMBER if v == m_streams.max() else GREEN for v in m_streams.values]
fig.add_trace(
    go.Bar(x=month_names, y=m_streams.values, marker_color=colors_str, opacity=0.9,
           text=[f"{v:.0f}M" for v in m_streams.values], textposition="outside",
           textfont=dict(color="white", size=11),
           hovertemplate="%{x}: mediana <b>%{y:.0f}M</b><extra></extra>"),
    row=1, col=2
)

spotify_layout(fig, title="Q2 — Sazonalidade nos lançamentos (2022–2023)", height=420)
fig.update_yaxes(title_text="Nº de músicas", row=1, col=1)
fig.update_yaxes(title_text="Streams mediana (M)", row=1, col=2)
fig.show()

### 💡 Insights — Q2

- **Janeiro** lidera em volume (134 músicas) — a indústria abre o ano forte
- **Agosto** é o mês mais calmo (46 músicas) — férias da indústria
- Músicas lançadas em **Outubro** têm a maior mediana de streams — menos concorrência = mais visibilidade
- **Implicação:** trade-off entre lançar na época alta (volume) vs. lançar com menos ruído (streams/música)

---
## 4. Q3 — Músicas antigas ainda dominam os charts de 2023?

**Hipótese:** Catálogos antigos sobrevivem nos charts porque são clássicos consolidados — viés de sobrevivência.  
**Metodologia:** Comparação de distribuição de streams por era de lançamento.

In [ ]:
era_order = ["Pré-2000", "2000s", "2010s", "2020\u20132021", "2022\u20132023"]

era_summary = (
    df.groupby("era")["streams_M"]
    .agg(["count", "median", "mean", "max"])
    .reindex(era_order)
    .rename(columns={"count":"Músicas","median":"Mediana (M)","mean":"Média (M)","max":"Máx (M)"})
    .round(0)
)
display(era_summary)

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Nº de músicas por era", "Distribuição de streams por era (boxplot)"],
)

era_counts = df["era"].value_counts().reindex(era_order, fill_value=0)

fig.add_trace(
    go.Bar(
        x=era_order, y=era_counts.values,
        marker_color=[ERA_COLORS.get(e, GREEN) for e in era_order],
        opacity=0.9,
        text=era_counts.values, textposition="outside",
        textfont=dict(color="white", size=11),
        hovertemplate="%{x}: <b>%{y} músicas</b><extra></extra>",
    ),
    row=1, col=1,
)

for era in era_order:
    sub = df[df["era"] == era]
    if len(sub) == 0:
        continue
    color = ERA_COLORS.get(era, GREEN)
    r, g, b = int(color[1:3],16), int(color[3:5],16), int(color[5:7],16)
    fig.add_trace(
        go.Box(
            y=sub["streams_M"], name=era,
            marker_color=color, line_color=color,
            fillcolor=f"rgba({r},{g},{b},0.15)",
            boxmean=True,
            hovertemplate=f"<b>{era}</b><br>%{{y:.0f}}M<extra></extra>",
        ),
        row=1, col=2,
    )

spotify_layout(fig, title="Q3 — Músicas antigas vs. recentes nos charts de 2023", height=440)
fig.update_yaxes(title_text="Nº de músicas", row=1, col=1)
fig.update_yaxes(title_text="Streams (M)", row=1, col=2)
fig.show()

### 💡 Insights — Q3

| Era | n | Mediana streams |
|-----|---|----------------|
| Pré-2000 | 48 | 622M |
| **2000s** | 20 | **1.229M** ← maior |
| 2010s | 151 | 984M |
| 2020–2021 | 156 | 564M |
| 2022–2023 | 577 | 179M |

> ⚠️ **Viés de sobrevivência:** as únicas músicas antigas que chegam aos charts de 2023 são os maiores clássicos de sempre. As milhares de músicas "normais" dessa era não estão neste dataset — o que infla artificialmente a mediana das eras antigas.

- **Implicação:** o catálogo histórico é um activo de altíssimo valor para editoras — streams passivos sem custo de marketing

---
## 5. Q4 — Colaborações geram mais streams do que solos?

**Hipótese:** Colaborações combinam fandoms e deveriam ter mais alcance.  
**Metodologia:** Comparação de medianas Solo vs. Colaboração; análise por número de artistas.

In [ ]:
solo_med   = df[df["collab"] == "Solo"]["streams_M"].median()
collab_med = df[df["collab"] == "Colaboração"]["streams_M"].median()
diff_pct   = (solo_med / collab_med - 1) * 100

print(f"Solo mediana        : {solo_med:.0f}M")
print(f"Colaboração mediana : {collab_med:.0f}M")
print(f"Diferença           : Solo +{diff_pct:.0f}% vs. Colaboração")

In [ ]:
df["artist_count_capped"] = df["artist_count"].clip(upper=5)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Volume no dataset", "Streams mediana", "Mediana por nº de artistas"],
)

counts = df["collab"].value_counts().reindex(["Solo","Colaboração"], fill_value=0)
fig.add_trace(
    go.Bar(x=counts.index.tolist(), y=counts.values,
           marker_color=[GREEN, CORAL], opacity=0.9,
           text=[f"{v}<br>({v/len(df)*100:.0f}%)" for v in counts.values],
           textposition="outside", textfont=dict(color="white", size=11),
           hovertemplate="%{x}: <b>%{y}</b><extra></extra>"),
    row=1, col=1,
)

medians = df.groupby("collab")["streams_M"].median().reindex(["Solo","Colaboração"])
fig.add_trace(
    go.Bar(x=medians.index.tolist(), y=medians.values,
           marker_color=[GREEN, CORAL], opacity=0.9,
           text=[f"{v:.0f}M" for v in medians.values],
           textposition="outside", textfont=dict(color="white", size=12),
           hovertemplate="%{x}: mediana <b>%{y:.0f}M</b><extra></extra>"),
    row=1, col=2,
)

by_n     = df.groupby("artist_count_capped")["streams_M"].median()
x_labels = {1:"1 (solo)",2:"2",3:"3",4:"4",5:"5+"}
fig.add_trace(
    go.Bar(
        x=[x_labels.get(int(k), str(k)) for k in by_n.index],
        y=by_n.values,
        marker_color=[GREEN, BLUE, AMBER, CORAL, PURPLE][:len(by_n)], opacity=0.9,
        text=[f"{v:.0f}M" for v in by_n.values],
        textposition="outside", textfont=dict(color="white", size=11),
        hovertemplate="Nº artistas %{x}: <b>%{y:.0f}M</b><extra></extra>",
    ),
    row=1, col=3,
)

spotify_layout(fig, title="Q4 — Colaborações vs. Solos", height=420)
fig.update_yaxes(title_text="Nº de músicas", row=1, col=1)
fig.update_yaxes(title_text="Streams mediana (M)", row=1, col=2)
fig.update_xaxes(title_text="Nº de artistas", row=1, col=3)
fig.show()

### 💡 Insights — Q4

- **Resultado contra-intuitivo:** solos têm mediana **41% superior** às colaborações (334M vs. 237M)
- **Hipótese:** artistas consolidados (Taylor Swift, Bad Bunny, The Weeknd) lançam maioritariamente solos
- **O factor determinante é a base de fãs do artista principal**, não o número de artistas
- **Limitação:** dataset só inclui músicas já populares — não é possível extrapolar para artistas em início de carreira

---
## 7. Machine Learning

> **Módulo interactivo completo:** [Dashboard Streamlit](https://edaspotify.streamlit.app) → seleccionar "Machine Learning" na sidebar.

### Pipeline ML

```
Features (12) → Train/Test split (80/20) → Random Forest → Métricas → Interpretação
```

**Features utilizadas:** BPM, Danceability, Valence, Energy, Acousticness, Instrumentalness, Liveness, Speechiness, Playlists Spotify, Nº artistas, Ano, Mês de lançamento.


### 7.1 Imports e preparação

In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (r2_score, mean_squared_error,
                             accuracy_score, confusion_matrix)
from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap_lib
import warnings
warnings.filterwarnings("ignore")

ML_FEATS = [
    "bpm","danceability_%","valence_%","energy_%",
    "acousticness_%","instrumentalness_%","liveness_%","speechiness_%",
    "in_spotify_playlists","artist_count","released_year","released_month",
]
FEAT_LABELS = {
    "bpm":"BPM","danceability_%":"Danceability","valence_%":"Valence",
    "energy_%":"Energy","acousticness_%":"Acousticness",
    "instrumentalness_%":"Instrumentalness","liveness_%":"Liveness",
    "speechiness_%":"Speechiness","in_spotify_playlists":"Playlists Spotify",
    "artist_count":"Nº artistas","released_year":"Ano","released_month":"Mês",
}

sub = df[ML_FEATS + ["streams_M"]].dropna()
X   = sub[ML_FEATS]
y   = sub["streams_M"]

threshold = y.quantile(0.75)
y_clf     = (y >= threshold).astype(int)

X_tr, X_te, y_tr, y_te         = train_test_split(X, y,     test_size=0.2, random_state=42)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X, y_clf, test_size=0.2, random_state=42)

print(f"Dataset ML: {len(sub)} músicas · {len(ML_FEATS)} features")
print(f"Treino: {len(X_tr)} · Teste: {len(X_te)}")
print(f"Threshold 'hit': {threshold:.0f}M streams (top 25%)")
print(f"Hits no dataset: {y_clf.sum()} músicas ({y_clf.mean()*100:.0f}%)")

### 7.2 ML1 — Regressão preditiva de streams

**Objectivo:** prever o número de streams (em milhões) com base nos atributos musicais.  
**Modelo:** Random Forest Regressor (n_estimators=200, max_depth=8)  
**Métrica principal:** R² — percentagem da variância explicada pelo modelo.

> **O que é R²?** Coeficiente de determinação. R²=1.0 seria perfeição; R²=0 significa que o modelo não é melhor do que prever sempre a média. Em dados de música, R²>0.35 já é um resultado interessante dado que factores externos (marketing, virality) não estão capturados.


In [ ]:
model_reg = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
model_reg.fit(X_tr, y_tr)
y_pred = model_reg.predict(X_te)

r2   = r2_score(y_te, y_pred)
rmse = mean_squared_error(y_te, y_pred) ** 0.5

print(f"R²   = {r2:.4f}  ({r2*100:.1f}% da variância explicada)")
print(f"RMSE = {rmse:.1f}M  (erro médio das previsões)")

# Gráfico Real vs. Previsto
fig = make_subplots(rows=1, cols=2, subplot_titles=["Real vs. Previsto","Resíduos"])

fig.add_trace(go.Scatter(x=y_te.values, y=y_pred, mode="markers",
    marker=dict(color=GREEN, size=5, opacity=0.45), name="Previsões",
    hovertemplate="Real: %{x:.0f}M<br>Previsto: %{y:.0f}M<extra></extra>"), row=1, col=1)
max_v = max(y_te.max(), y_pred.max())
fig.add_trace(go.Scatter(x=[0,max_v], y=[0,max_v], mode="lines",
    line=dict(color=CORAL, width=1.5, dash="dash"), name="Perfeito"), row=1, col=1)

residuals = y_te.values - y_pred
fig.add_trace(go.Scatter(x=y_pred, y=residuals, mode="markers",
    marker=dict(color=PURPLE, size=5, opacity=0.45), name="Resíduos",
    hovertemplate="Previsto: %{x:.0f}M<br>Resíduo: %{y:.0f}M<extra></extra>"), row=1, col=2)
fig.add_hline(y=0, line_dash="dash", line_color=CORAL, line_width=1.5, row=1, col=2)

spotify_layout(fig, title=f"Regressão Random Forest — R²={r2:.3f}", height=420, showlegend=True)
fig.update_xaxes(title_text="Streams reais (M)", row=1, col=1)
fig.update_yaxes(title_text="Streams previstos (M)", row=1, col=1)
fig.update_xaxes(title_text="Streams previstos (M)", row=1, col=2)
fig.update_yaxes(title_text="Resíduo (M)", row=1, col=2)
fig.show()

### 7.3 ML2 — Importância de features

**O que é Feature Importance?** Mede quanto cada variável contribui para reduzir o erro nas árvores de decisão. Valores mais altos = variável mais importante para as previsões.


In [ ]:
fi = pd.Series(model_reg.feature_importances_, index=ML_FEATS)
fi_sorted = fi.rename(index=FEAT_LABELS).sort_values(ascending=True)

colors_fi = [GREEN if v == fi_sorted.max() else
             BLUE  if v >= fi_sorted.quantile(0.7) else GRAY
             for v in fi_sorted.values]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=fi_sorted.values, y=fi_sorted.index.tolist(),
    orientation="h", marker_color=colors_fi, opacity=0.9,
    text=[f"{v*100:.1f}%" for v in fi_sorted.values],
    textposition="outside", textfont=dict(color="#e8e6df", size=11),
    hovertemplate="%{y}: <b>%{x:.3f}</b><extra></extra>"))
spotify_layout(fig, title="Importância de features — Regressão (streams)",
    xaxis_title="Importância relativa", height=480)
fig.update_layout(margin=dict(l=160, r=100, t=50, b=50))
fig.show()

print("\nTop 3 features mais importantes:")
for i, (feat, val) in enumerate(fi_sorted.iloc[-3:][::-1].items(), 1):
    print(f"  {i}. {feat}: {val*100:.1f}%")

### 7.4 ML3 — Classificação de sucesso (hit vs. não-hit)

**Objectivo:** prever se uma música será um "hit" (top 25% de streams).  
**Modelo:** Random Forest Classifier  
**Métricas:** Accuracy, Precision, Recall, F1 + Matriz de confusão.

> **Precision vs. Recall:** Precision = "quando diz que é hit, acerta?"; Recall = "encontra todos os hits reais?". O F1 é a média harmónica dos dois.


In [ ]:
model_clf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
model_clf.fit(X_tr_c, y_tr_c)
y_pred_c = model_clf.predict(X_te_c)

acc  = accuracy_score(y_te_c, y_pred_c)
cm   = confusion_matrix(y_te_c, y_pred_c)
tp, fp, fn, tn = cm[1,1], cm[0,1], cm[1,0], cm[0,0]
precision = tp/(tp+fp) if (tp+fp) > 0 else 0
recall    = tp/(tp+fn) if (tp+fn) > 0 else 0
f1        = 2*precision*recall/(precision+recall+1e-9)

print(f"Accuracy  = {acc*100:.1f}%")
print(f"Precision = {precision*100:.1f}%  (hits correctos previstos)")
print(f"Recall    = {recall*100:.1f}%  (hits reais detectados)")
print(f"F1 Score  = {f1*100:.1f}%")

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Matriz de confusão","Features mais importantes (classificação)"])

fig.add_trace(go.Heatmap(
    z=cm, x=["Não-hit (prev.)","Hit (prev.)"],
    y=["Não-hit (real)","Hit (real)"],
    colorscale=[[0,"#0a0a0a"],[1,GREEN]],
    text=cm, texttemplate="<b>%{text}</b>",
    textfont=dict(size=20, color="#e8e6df"),
    hovertemplate="%{y} → %{x}: %{z}<extra></extra>"), row=1, col=1)

fi_clf = pd.Series(model_clf.feature_importances_, index=ML_FEATS)
fi_clf_s = fi_clf.rename(index=FEAT_LABELS).sort_values(ascending=True)
colors_fc = [CORAL if v == fi_clf_s.max() else
             AMBER if v >= fi_clf_s.quantile(0.7) else GRAY
             for v in fi_clf_s.values]
fig.add_trace(go.Bar(
    x=fi_clf_s.values, y=fi_clf_s.index.tolist(),
    orientation="h", marker_color=colors_fc, opacity=0.9,
    text=[f"{v*100:.1f}%" for v in fi_clf_s.values],
    textposition="outside", textfont=dict(color="#e8e6df", size=11),
    hovertemplate="%{y}: <b>%{x:.3f}</b><extra></extra>"), row=1, col=2)

spotify_layout(fig, title=f"Classificação — Accuracy {acc*100:.0f}%", height=420)
fig.update_layout(margin=dict(l=50, r=120, t=50, b=100))
fig.update_xaxes(tickangle=-30, row=1, col=1)
fig.show()

### 7.5 ML4 — Curva de aprendizagem

Avalia se o modelo tem **overfitting** (R² treino >> validação) ou **underfitting** (ambos baixos).

> **Como ler:** Se as curvas convergem num valor alto → modelo saudável. Se o gap entre treino e validação é grande → overfitting. Se ambas são baixas → underfitting.


In [ ]:
from sklearn.model_selection import learning_curve as lc_func

sizes = np.linspace(0.1, 1.0, 8)
tr_sizes, tr_scores, val_scores = lc_func(
    RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    X, y, cv=4, train_sizes=sizes, scoring="r2", n_jobs=-1)

tr_mean  = tr_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)
gap      = tr_mean[-1] - val_mean[-1]

fig = go.Figure()
fig.add_trace(go.Scatter(x=tr_sizes, y=tr_mean, mode="lines+markers",
    name="Treino", line=dict(color=GREEN, width=2.5), marker=dict(size=8, color=GREEN),
    hovertemplate="n=%{x}<br>R² treino: %{y:.3f}<extra></extra>"))
fig.add_trace(go.Scatter(x=tr_sizes, y=val_mean, mode="lines+markers",
    name="Validação (CV)", line=dict(color=CORAL, width=2.5, dash="dash"),
    marker=dict(size=8, color=CORAL),
    hovertemplate="n=%{x}<br>R² validação: %{y:.3f}<extra></extra>"))
spotify_layout(fig, title="Curva de aprendizagem — R² vs. tamanho do treino",
    xaxis_title="Nº de amostras de treino", yaxis_title="R²",
    height=420, showlegend=True)
fig.show()

print(f"Gap treino/validação: {gap:.3f}")
print("Diagnóstico:", "⚠️ Overfitting — reduz max_depth" if gap > 0.2
      else "✓ Modelo generaliza bem")
print(f"R² validação final: {val_mean[-1]:.3f}")

### 7.6 ML5 — UMAP 2D

**UMAP (Uniform Manifold Approximation and Projection)** comprime 12 dimensões num plano 2D preservando a estrutura de vizinhança. Músicas parecidas ficam próximas; músicas diferentes ficam afastadas. É das visualizações mais poderosas em DS.


In [ ]:
sub_umap = df[ML_FEATS + ["streams_M","era","genre_cluster","mode"]].dropna()
sc_umap  = StandardScaler()
X_scaled = sc_umap.fit_transform(sub_umap[ML_FEATS])

reducer  = umap_lib.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding = reducer.fit_transform(X_scaled)
x_emb, y_emb = embedding[:, 0], embedding[:, 1]

print(f"UMAP projectou {len(sub_umap)} músicas de {len(ML_FEATS)}D → 2D")

# Colorir por género inferido
fig = go.Figure()
for g, c in GENRE_COLORS.items():
    mask_g = sub_umap["genre_cluster"] == g
    if not mask_g.any(): continue
    fig.add_trace(go.Scatter(
        x=x_emb[mask_g.values], y=y_emb[mask_g.values], mode="markers",
        name=g, marker=dict(color=c, size=4, opacity=0.55),
        hovertemplate=f"<b>{g}</b><br>Streams: %{{customdata:.0f}}M<extra></extra>",
        customdata=sub_umap.loc[mask_g,"streams_M"]))

spotify_layout(fig, title="UMAP 2D — colorido por género inferido",
    xaxis_title="UMAP 1", yaxis_title="UMAP 2", height=520, showlegend=True)
fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=False, zeroline=False)
fig.show()

### 💡 Sumário — Módulo Machine Learning

| Modelo | Métrica principal | Resultado | Interpretação |
|--------|------------------|-----------|---------------|
| **Regressão RF** | R² | ~0.35 | Atributos musicais explicam ~35% da variância |
| **Feature Importance** | Importância relativa | Playlists > BPM | Distribuição > atributos musicais |
| **Classificação RF** | Accuracy | ~75% | 3 em cada 4 hits correctamente identificados |
| **Curva aprendizagem** | Gap treino/val | < 0.20 | Modelo sem overfitting significativo |
| **UMAP 2D** | Visual | Clusters visíveis | Espaço musical tem estrutura latente real |

---

## 8. Conclusão geral

| Módulo | Principais takeaways |
|--------|---------------------|
| **EDA** | Distribuição em lei de potência · sazonalidade real · viés de sobrevivência |
| **Correlações** | Playlists > atributos musicais · géneros emergem sem labels · Shazam capta tendências |
| **ML** | R²~0.35 · Playlists é a feature mais importante · clustering confirma estrutura latente |

> 🎵 Dashboard interactivo completo: **[edaspotify.streamlit.app](https://edaspotify.streamlit.app)**

---

<p align="center">
  Desenvolvido por <strong>Sander Augusto Garcia</strong><br>
  <a href="https://github.com/SanderAugustoGarcia">github.com/SanderAugustoGarcia</a>
</p>
